# MorphoGAT: paired training and evaluation

Install the project dependencies in this notebook's Python environment first.
This walkthrough uses simulated sections to verify the workflow.
Use new output directories when rerunning cells; existing experiment files are protected.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "morphogat").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "morphogat").is_dir(), "Open this notebook inside the source checkout"

def run(*arguments):
    subprocess.run([sys.executable, "-m", "morphogat", *map(str, arguments)], cwd=ROOT, check=True)


## Make and validate the demonstration sections

The real-data workflow uses the same manifest format with raw counts and registered tissue images.

In [ ]:
DATA = ROOT / "data" / "notebook_demo"
MODEL = ROOT / "runs" / "notebook_model"
TEST = ROOT / "runs" / "notebook_test"
run("make-demo", "--output", DATA)
run("check-data", "--manifest", DATA / "manifest.yaml", "--config", DATA / "config.yaml",
    "--output", ROOT / "runs" / "notebook_check")


## Fit the paired model

The small demo runs four CPU epochs. For study data use `configs/reference.yaml` and an explicit validation partition.

In [ ]:
run("train", "--manifest", DATA / "manifest.yaml", "--config", DATA / "config.yaml",
    "--output", MODEL, "--device", "cpu")


## Evaluate the independent RNA-to-protein path

Held-out proteins are used as targets only. The auxiliary protein encoder is not called.

In [ ]:
run("evaluate", "--checkpoint", MODEL / "best.pt", "--manifest", DATA / "manifest.yaml",
    "--split", "test", "--output", TEST, "--device", "cpu")
import pandas as pd
pd.read_csv(TEST / "demo_3" / "protein_metrics.csv")


In [ ]:
import matplotlib.pyplot as plt
history = pd.read_csv(MODEL / "history.csv")
history.plot(x="epoch", y=["train_prediction", "validation_rmse"], marker="o")
plt.ylabel("Normalized protein RMSE")
plt.show()


## Study experiments

See `docs/EXPERIMENTS.md` for fixed-split repetitions, the four ablations, baseline-table evaluation and manuscript-style boxplots. The demo's numerical scores are not study results.